# Shape of Distributions
**Topic:** Descriptive Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** what skewness measures and which direction a distribution leans
- **Explain** how kurtosis captures the heaviness of a distribution's tails
- **Interpret** a histogram's shape to identify whether data is symmetric, skewed, or bimodal

> **Tip:** Start by moving the **skewness slider** toward positive values and watch the long tail stretch to the right, then try negative values to flip the lean.

---
## How we got here

In *04: Central Tendency* and *05: Dispersion* we captured a distribution with just two numbers: where it's centered and how wide it is. But two datasets can share the same mean and standard deviation while looking completely different, one could be symmetric, the other lopsided. This notebook adds the third and fourth moments to complete the picture.

---
## Why this matters for data science

Many ML algorithms assume features are normally distributed (symmetric, no heavy tails). When that assumption is violated, models behave unpredictably. Log-transforming a right-skewed feature before feeding it to a linear model is a direct response to skewness. Detecting heavy tails (high kurtosis) in residuals tells you your model is missing something for extreme cases.

---
## Try it yourself

In [2]:

# ─── State ────────────────────────────────────────────────────────────────────

_N = 4000

PRESETS = {
    "Normal":       {"skewness":  0.0,  "kurtosis":  0.0},
    "Right-Skewed": {"skewness":  2.0,  "kurtosis":  0.5},
    "Left-Skewed":  {"skewness": -2.0,  "kurtosis":  0.5},
    "Bimodal":      {"skewness":  0.0,  "kurtosis": -0.75},
}

_updating_sliders = False   # suppress slider observers during preset snap
_locked_preset    = None    # "Bimodal" → special generation

# ─── Widgets ──────────────────────────────────────────────────────────────────

out = Output()

skew_slider = FloatSlider(
    value=0.0, min=-3.0, max=3.0, step=0.25,
    description="Skewness:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
kurt_slider = FloatSlider(
    value=0.0, min=-2.0, max=5.0, step=0.25,
    description="Kurtosis:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)
preset_dd = Dropdown(
    options=["Normal", "Right-Skewed", "Left-Skewed", "Bimodal"],
    value="Normal",
    description="Preset:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="420px"),
)

# ─── Sample generation ────────────────────────────────────────────────────────

def _get_samples(skewness, kurtosis):
    rng = np.random.default_rng(42)
    if _locked_preset == "Bimodal":
        return np.concatenate([
            rng.normal(-2.0, 0.8, _N // 2),
            rng.normal( 2.0, 0.8, _N // 2),
        ])
    # skew-normal base
    a    = skewness * 4.0
    base = stats.skewnorm.rvs(a=a, size=_N, random_state=42)
    z    = (base - base.mean()) / (base.std() + 1e-10)
    if kurtosis > 0.05:
        # Leptokurtic: power stretch sharpens the peak and fattens the tails.
        k = 1.0 + kurtosis * 0.25
        z = np.sign(z) * np.abs(z) ** k
        z = (z - z.mean()) / (z.std() + 1e-10)
    elif kurtosis < -0.05:
        # Platykurtic: add uniform noise WITHOUT re-standardizing.
        # Leaving the spread intact preserves the "wide and flat" shape —
        # the same way leptokurtic naturally looks "narrow and tall" after its
        # power transform. Re-standardizing would compress the flatness away.
        delta = abs(kurtosis) * 1.6
        z = z + rng.uniform(-delta, delta, _N)
    return z

# ─── Summary helpers ──────────────────────────────────────────────────────────

def _order_label(mean, median, mode, std):
    tol   = std * 0.10
    items = sorted(
        [("mean", mean), ("median", median), ("mode", mode)],
        key=lambda t: t[1],
    )
    parts, group = [], [items[0][0]]
    for i in range(1, 3):
        if abs(items[i][1] - items[i - 1][1]) < tol:
            group.append(items[i][0])
        else:
            parts.append(" ≈ ".join(group))
            group = [items[i][0]]
    parts.append(" ≈ ".join(group))
    return " < ".join(parts)

def _tail_sentence(skew, kurt):
    if _locked_preset == "Bimodal":
        return (
            "Two peaks indicate two underlying groups — "
            "skewness and kurtosis alone do not fully describe a bimodal shape."
        )
    skew_txt = (
        "roughly symmetric"
        if abs(skew) < 0.3
        else "right-skewed — the tail stretches toward higher values"
        if skew > 0
        else "left-skewed — the tail stretches toward lower values"
    )
    kurt_txt = (
        "with heavy tails and more extreme outliers than normal (leptokurtic)"
        if kurt > 1.5
        else "with lighter tails and fewer extreme values than normal (platykurtic)"
        if kurt < -0.5
        else "with tail weight close to a normal distribution (mesokurtic)"
    )
    return f"This distribution is {skew_txt}, {kurt_txt}."

# ─── Render ───────────────────────────────────────────────────────────────────

def render():
    skew = skew_slider.value
    kurt = kurt_slider.value
    data = _get_samples(skew, kurt)

    mean   = float(np.mean(data))
    median = float(np.median(data))
    std    = float(np.std(data))
    a_skew = float(stats.skew(data))
    a_kurt = float(stats.kurtosis(data))   # excess kurtosis

    # KDE and mode
    x_lo, x_hi = data.min() - std * 0.5, data.max() + std * 0.5
    x_kde = np.linspace(x_lo, x_hi, 500)
    kde   = stats.gaussian_kde(data)
    y_kde = kde(x_kde)
    mode  = float(x_kde[np.argmax(y_kde)])
    y_top = float(y_kde.max()) * 1.08

    def vline(x_val, color, label):
        return go.Scatter(
            x=[x_val, x_val], y=[0, y_top],
            mode="lines",
            line=dict(color=color, width=2, dash="dash"),
            name=label,
        )

    traces = [
        go.Histogram(
            x=data, nbinsx=50, histnorm="probability density",
            marker_color=PALETTE["primary"], opacity=0.40,
            name="Sample distribution",
        ),
        go.Scatter(
            x=x_kde, y=y_kde, mode="lines",
            line=dict(color=PALETTE["primary"], width=2.5),
            name="KDE",
        ),
        vline(mean,   PALETTE["secondary"], f"Mean = {mean:.2f}"),
        vline(median, PALETTE["accent"],    f"Median = {median:.2f}"),
        vline(mode,   PALETTE["muted"],     f"Mode ≈ {mode:.2f}"),
    ]

    fig_layout = base_layout(
        title="Shape of Distributions Explorer",
        xaxis_title="Value",
        yaxis_title="Probability Density",
    )
    fig_layout.update(showlegend=True)
    fig = go.Figure(data=traces, layout=fig_layout)

    order = _order_label(mean, median, mode, std)
    tail  = _tail_sentence(skew, kurt)
    html  = f"""
<div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:6px;
            padding:10px 16px;margin-top:6px;font-size:14px;font-family:sans-serif;">
  <table style="width:100%;border-collapse:collapse;">
    <tr>
      <td style="padding:4px 12px;color:#555;font-weight:bold;">Skewness</td>
      <td style="padding:4px 12px;">{a_skew:+.2f}</td>
      <td style="padding:4px 12px;color:#555;font-weight:bold;">Excess kurtosis</td>
      <td style="padding:4px 12px;">{a_kurt:+.2f}</td>
    </tr>
    <tr>
      <td style="padding:4px 12px;color:#555;font-weight:bold;">Order</td>
      <td colspan="3" style="padding:4px 12px;">{order}</td>
    </tr>
    <tr>
      <td style="padding:4px 12px;color:#555;font-weight:bold;">Tail behavior</td>
      <td colspan="3" style="padding:4px 12px;">{tail}</td>
    </tr>
  </table>
</div>"""

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))
        display(HTML(html))

# ─── Observers ────────────────────────────────────────────────────────────────

def _on_preset(change):
    global _updating_sliders, _locked_preset
    preset = change["new"]
    _locked_preset    = "Bimodal" if preset == "Bimodal" else None
    _updating_sliders = True
    p = PRESETS[preset]
    skew_slider.value = p["skewness"]
    kurt_slider.value = p["kurtosis"]
    _updating_sliders = False
    render()

def _on_slider(change):
    global _locked_preset
    if _updating_sliders:
        return
    _locked_preset = None
    render()

skew_slider.observe(_on_slider, names="value")
kurt_slider.observe(_on_slider, names="value")
preset_dd.observe(_on_preset,   names="value")

# ─── Display (once) ───────────────────────────────────────────────────────────

widget_box = VBox([VBox([skew_slider, kurt_slider, preset_dd]), out])

if not hasattr(out, '_initialized'):
    display(widget_box)
    out._initialized = True

render()


---
## What's happening?

Shape statistics go beyond center and spread to describe the **geometry** of the distribution.

| Statistic | Symbol | What it measures |
|-----------|--------|-----------------|
| Skewness | γ₁ | Asymmetry: positive means right tail is longer |
| Kurtosis | γ₂ | Tail heaviness: high means more extreme outliers |
| Excess kurtosis | γ₂ − 3 | Kurtosis relative to the normal distribution (which has γ₂ = 3) |

$$\text{Skewness} = \frac{1}{n} \sum_{i=1}^{n} \left(\frac{x_i - \bar{x}}{s}\right)^3$$

$$\text{Kurtosis} = \frac{1}{n} \sum_{i=1}^{n} \left(\frac{x_i - \bar{x}}{s}\right)^4$$

### The mean–median relationship reveals skew

When a distribution is right-skewed, the mean gets pulled toward the long tail and ends up **above** the median. When left-skewed, the mean falls **below** the median. A perfectly symmetric distribution has mean equal to median equal to mode.

Check this against the widget: drag toward positive skew and verify that mean > median in the output.

---
## Real-world example: Home sale prices in a metropolitan area

Housing prices are almost universally right-skewed. Most homes sell in a moderate range, but a small number of luxury properties create a long right tail that pulls the mean well above the median, exactly the pattern we saw with income in the previous notebook.

The chart below shows simulated home sale prices for a major metro area using parameters calibrated to real 2023 market data. Notice:

- **Notice:** The bulk of sales cluster below $600k, but the distribution has a long rightward tail stretching past $2M
- **Notice:** The mean is well above the median, a concrete marker of positive skew visible directly in the chart
- **Notice:** The KDE curve is not symmetric around its peak, the right side descends more slowly than the left

> **Discussion question:** A developer is targeting the "typical" buyer. Which statistic would give a more accurate picture of what most buyers can afford, the mean sale price, the median, or the modal price range? Does the answer change depending on how you define "typical"?

In [3]:
np.random.seed(14)

# ── Simulated home prices — right-skewed lognormal (calibrated to 2023 US metro data) ──
n       = 1500
prices  = np.random.lognormal(mean=13.0, sigma=0.55, size=n)  # roughly $440k median
prices  = prices / 1000  # express in $000s
prices  = np.clip(prices, 100, 3500)

mean_p   = np.mean(prices)
median_p = np.median(prices)
skew_p   = float(stats.skew(prices))

x_kde = np.linspace(prices.min(), prices.max(), 500)
kde   = stats.gaussian_kde(prices)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=prices, nbinsx=55, histnorm="probability density",
    marker_color=PALETTE["primary"], opacity=0.55, name="Sale prices",
))
fig.add_trace(go.Scatter(
    x=x_kde, y=kde(x_kde),
    mode="lines", line=dict(color=PALETTE["primary"], width=2.5),
    name="KDE",
))
fig.add_vline(x=mean_p,   line_color=PALETTE["secondary"], line_width=2,
              annotation_text=f"Mean ${mean_p:.0f}k")
fig.add_vline(x=median_p, line_color=PALETTE["accent"], line_width=2,
              annotation_text=f"Median ${median_p:.0f}k")
layout = base_layout(
    title=f"Home Sale Prices — Right-Skewed (skewness = {skew_p:.2f})",
    xaxis_title="Sale Price ($000s)",
    yaxis_title="Probability Density",
)
layout.update(xaxis=dict(range=[0, 2200]))
fig.update_layout(layout)
fig.show()

### Common distribution shapes in real-world data

| Shape | Skewness | Example | Why it occurs |
|-------|----------|---------|--------------|
| Symmetric (normal) | ≈ 0 | Adult heights, IQ scores | Many independent additive factors |
| Right-skewed (positive) | > 0 | Income, home prices, wait times | Lower bound at zero, no upper bound |
| Left-skewed (negative) | < 0 | Age at retirement, exam scores (easy test) | Upper bound, few very low values |
| Bimodal | ≈ 0 (can mislead) | Heights of mixed-sex sample, commute times | Two underlying populations mixed together |
| Heavy-tailed (high kurtosis) | ≈ 0 | Daily stock returns | Rare extreme events more common than normal predicts |

---
## Key takeaway

> **Skewness tells you which direction the tail stretches; kurtosis tells you how heavy that tail is — together they reveal whether the normal distribution is a safe assumption for your data.**

---
*Next up: Correlation & Covariance — what happens when we measure the relationship between two variables instead of one?*